# 03 - Model Serving with KServe

![Inference Workflow](../../docs/diagrams/03-inference-workflow.png)

## Overview

This notebook deploys the trained model as a **KServe InferenceService** for real-time predictions. The key feature is **Feast integration** - clients send only entity IDs, and the server fetches features from Feast's online store.

### What You'll Learn

| Step | Action | Component |
|------|--------|-----------|
| 1 | Create serving script | ConfigMap with KServe Model class |
| 2 | Deploy InferenceService | KServe with Feast remote client |
| 3 | Make predictions | REST API (V2 protocol) |

### Inference Flow

| Step | Component | Action |
|------|-----------|--------|
| 1 | Client | Sends entity IDs `{store_id, dept_id}` |
| 2 | KServe | Receives request |
| 3 | Feast SDK | Fetches features via gRPC (operator-managed) |
| 4 | Model | Scales features, runs inference |
| 5 | KServe | Returns prediction `{weekly_sales: $96,763}` |

### Why Feast Integration?

| Approach | Client Sends | Pros | Cons |
|----------|--------------|------|------|
| **Without Feast** | All 21 features | Simple server | Feature drift, complex client |
| **With Feast** | Entity IDs only | No drift, simple client | Requires Feast online store |

**Key Benefit**: Training and serving use the **same FeatureService**, eliminating feature skew.

### Prerequisites

1. Features registered and materialized (run `01a-local.ipynb` or `01b-remote.ipynb`)
2. Model trained (run `02-training.ipynb`)
3. KServe operator installed on cluster

---
## Step 0: Install Dependencies

In [ ]:
%pip install -q kserve pandas requests kubernetes

In [ ]:
from kserve import KServeClient, V1beta1InferenceService, V1beta1InferenceServiceSpec, V1beta1PredictorSpec
from kubernetes import client as k8s_client, config
from kubernetes.client import (
    V1Container, V1ResourceRequirements, V1VolumeMount, V1Volume,
    V1PersistentVolumeClaimVolumeSource, V1ConfigMapVolumeSource,
    V1EnvVar, V1Probe, V1HTTPGetAction, V1ConfigMap, V1ObjectMeta, V1KeyToPath
)
import pandas as pd
import requests
import time
import json
from datetime import datetime

---
## Step 1: Configuration

| Parameter | Value | Purpose |
|-----------|-------|---------|
| `MODEL_NAME` | sales-forecast | InferenceService name |
| `MODEL_DIR` | /shared/models | Path to model artifacts |
| `FEAST_CONFIG_PATH` | Operator-provided | Feast remote client config (gRPC) |

In [ ]:
NAMESPACE = "feast-trainer-demo"
MODEL_NAME = "sales-forecast"
MODEL_DIR = "/shared/models"
PVC_NAME = "shared"
CONFIGMAP_NAME = "sales-forecast-serve"

# Trainer image (has PyTorch pre-installed)
TRAINER_IMAGE = "quay.io/modh/training:py312-cuda128-torch280"

# Feast remote client config (operator-provided ConfigMaps)
FEAST_CONFIG_PATH = "/opt/app-root/src/feast-config/salesforecasting"
FEAST_CONFIG_CM = "feast-salesforecasting-client"
FEAST_CA_CM = "feast-salesforecasting-client-ca"

print(f"📋 Configuration:")
print(f"   Namespace: {NAMESPACE}")
print(f"   Model: {MODEL_NAME}")
print(f"   Image: {TRAINER_IMAGE}")
print(f"   PVC: {PVC_NAME}")
print(f"   Feast Config: {FEAST_CONFIG_PATH}")

---
## Step 2: Load Serving Script

The serving script is defined in `serving_script.py` (same directory). It implements KServe's `Model` interface:

| Method | Purpose | Called When |
|--------|---------|-------------|
| `load()` | Load model, scalers, Feast store | Server startup |
| `preprocess()` | Extract entities → Feast SDK → Build feature matrix | Each request |
| `predict()` | Scale features → Model inference → Inverse transform | Each request |
| `postprocess()` | Format as V2 InferResponse | Each request |

### Feast Integration

Uses **Feast SDK directly** with the operator-provided remote client config:
- Mounts `feast-salesforecasting-client` ConfigMap
- Calls `store.get_online_features()` via gRPC
- Same approach as training (no REST API needed)

In [ ]:
# Load serving script from file
from pathlib import Path

script_path = Path("serving_script.py")
if script_path.exists():
    SERVE_SCRIPT = script_path.read_text()
    print(f"✅ Loaded serving script from {script_path}")
    print(f"   Lines: {len(SERVE_SCRIPT.splitlines())}")
else:
    raise FileNotFoundError(f"serving_script.py not found in {Path.cwd()}")

In [ ]:
# Load Kubernetes config
try:
    config.load_incluster_config()
    print("✅ Loaded in-cluster Kubernetes config")
except:
    config.load_kube_config()
    print("✅ Loaded local Kubernetes config")

core_v1 = k8s_client.CoreV1Api()

In [ ]:
# Create ConfigMap with serving script
configmap = V1ConfigMap(
    metadata=V1ObjectMeta(
        name=CONFIGMAP_NAME,
        namespace=NAMESPACE,
        labels={"app": "sales-forecasting"}
    ),
    data={"serve.py": SERVE_SCRIPT}
)

try:
    core_v1.read_namespaced_config_map(CONFIGMAP_NAME, NAMESPACE)
    print(f"⚠️ ConfigMap '{CONFIGMAP_NAME}' exists, replacing...")
    core_v1.replace_namespaced_config_map(CONFIGMAP_NAME, NAMESPACE, configmap)
except:
    print(f"📦 Creating ConfigMap '{CONFIGMAP_NAME}'...")
    core_v1.create_namespaced_config_map(NAMESPACE, configmap)

print(f"✅ ConfigMap created with serve.py script")

---
## Step 3: Deploy InferenceService

### InferenceService Configuration

| Setting | Value | Purpose |
|---------|-------|---------|
| `minReplicas` | 1 | Always have at least 1 replica |
| `maxReplicas` | 3 | Scale up under load |
| `deploymentMode` | RawDeployment | Use standard K8s deployment |
| `readinessProbe` | /v2/health/ready | Health check endpoint |

In [ ]:
kserve_client = KServeClient()
print(f"✅ KServeClient initialized")

In [ ]:
# Build InferenceService spec
isvc = V1beta1InferenceService(
    api_version="serving.kserve.io/v1beta1",
    kind="InferenceService",
    metadata=k8s_client.V1ObjectMeta(
        name=MODEL_NAME,
        namespace=NAMESPACE,
        labels={"app": "sales-forecasting"},
        annotations={"serving.kserve.io/deploymentMode": "RawDeployment"}
    ),
    spec=V1beta1InferenceServiceSpec(
        predictor=V1beta1PredictorSpec(
            min_replicas=1,
            max_replicas=3,
            containers=[
                V1Container(
                    name="kserve-container",
                    image=TRAINER_IMAGE,
                    command=["/bin/bash", "-c"],
                    args=[
                        "pip install -q kserve joblib numpy scikit-learn pandas feast==0.61.0 psycopg2-binary && "
                        "python /scripts/serve.py"
                    ],
                    ports=[k8s_client.V1ContainerPort(container_port=8080, protocol="TCP")],
                    env=[
                        V1EnvVar(name="MODEL_DIR", value=MODEL_DIR),
                        V1EnvVar(name="FEAST_CONFIG_PATH", value=FEAST_CONFIG_PATH),
                    ],
                    resources=V1ResourceRequirements(
                        requests={"cpu": "500m", "memory": "2Gi"},
                        limits={"cpu": "2", "memory": "4Gi"}
                    ),
                    readiness_probe=V1Probe(
                        http_get=V1HTTPGetAction(path="/v2/health/ready", port=8080),
                        initial_delay_seconds=90,
                        period_seconds=10
                    ),
                    volume_mounts=[
                        V1VolumeMount(name="scripts", mount_path="/scripts"),
                        V1VolumeMount(name="model-storage", mount_path="/shared"),
                        V1VolumeMount(name="feast-config", mount_path=FEAST_CONFIG_PATH),
                        V1VolumeMount(name="feast-ca", mount_path="/etc/pki/tls/custom-certs"),
                    ]
                )
            ],
            volumes=[
                V1Volume(name="scripts", config_map=V1ConfigMapVolumeSource(name=CONFIGMAP_NAME)),
                V1Volume(name="model-storage", persistent_volume_claim=V1PersistentVolumeClaimVolumeSource(claim_name=PVC_NAME)),
                V1Volume(name="feast-config", config_map=V1ConfigMapVolumeSource(name=FEAST_CONFIG_CM)),
                V1Volume(
                    name="feast-ca",
                    config_map=V1ConfigMapVolumeSource(
                        name=FEAST_CA_CM,
                        items=[k8s_client.V1KeyToPath(key="service-ca.crt", path="ca-bundle.crt")]
                    )
                ),
            ]
        )
    )
)

print("✅ InferenceService spec created")

In [ ]:
# Deploy InferenceService
try:
    existing = kserve_client.get(MODEL_NAME, namespace=NAMESPACE)
    print(f"⚠️ InferenceService '{MODEL_NAME}' exists, replacing...")
    kserve_client.replace(MODEL_NAME, isvc, namespace=NAMESPACE)
except Exception:
    print(f"📦 Creating InferenceService '{MODEL_NAME}'...")
    kserve_client.create(isvc)

print(f"✅ InferenceService submitted to namespace '{NAMESPACE}'")

In [ ]:
# Wait for ready
print("⏳ Waiting for InferenceService to be ready...")
print("   This may take 1-2 minutes for the first deployment.")

try:
    kserve_client.wait_isvc_ready(MODEL_NAME, namespace=NAMESPACE, timeout_seconds=300)
    isvc_status = kserve_client.get(MODEL_NAME, namespace=NAMESPACE)
    url = isvc_status.get("status", {}).get("url", "")
    print(f"✅ InferenceService ready!")
    print(f"   URL: {url}")
except Exception as e:
    print(f"⚠️ Timeout waiting for ready: {e}")
    print("   Check pod status: kubectl get pods -n feast-trainer-demo -l app=sales-forecasting")

---
## Step 4: Make Predictions (V2 Protocol)

### V2 Inference Protocol

KServe uses the V2 inference protocol. Request format:

```json
{
  "inputs": [
    {
      "name": "entities",
      "shape": [n],
      "datatype": "BYTES",
      "data": [{"store_id": 1, "dept_id": 3}, ...]
    }
  ]
}
```

Response format:

```json
{
  "outputs": [
    {
      "name": "predictions",
      "shape": [n],
      "datatype": "FP32",
      "data": [96763.45, ...]
    }
  ]
}
```

In [ ]:
# Construct endpoint URL
ENDPOINT = f"http://{MODEL_NAME}-predictor.{NAMESPACE}.svc.cluster.local:8080"

# Check health
try:
    resp = requests.get(f"{ENDPOINT}/v2/health/ready", timeout=5)
    ready = resp.status_code == 200
except:
    ready = False
    
print(f"Inference Endpoint: {ENDPOINT}")
print(f"Ready: {'✅ Yes' if ready else '❌ No'}")

In [ ]:
def predict_with_feast(entities: list) -> dict:
    """Make prediction using KServe V2 protocol with Feast feature lookup.
    
    Args:
        entities: List of entity dicts, e.g., [{"store_id": 1, "dept_id": 3}]
    
    Returns:
        Dict with predictions, latency, and entities
    """
    url = f"{ENDPOINT}/v2/models/{MODEL_NAME}/infer"
    payload = {
        "inputs": [
            {
                "name": "entities",
                "shape": [len(entities)],
                "datatype": "BYTES",
                "data": entities
            }
        ]
    }
    
    t0 = time.time()
    response = requests.post(url, json=payload, timeout=30)
    latency = (time.time() - t0) * 1000
    
    if response.status_code != 200:
        raise Exception(f"Prediction failed: {response.text}")
    
    result = response.json()
    predictions = result.get("outputs", [{}])[0].get("data", [])
    
    return {
        "predictions": predictions,
        "latency_ms": latency,
        "entities": entities
    }

In [ ]:
# Single prediction
print("📊 Single Prediction Test")
print("-" * 50)

entity = {"store_id": 1, "dept_id": 3}
try:
    result = predict_with_feast([entity])
    print(f"✅ Store {entity['store_id']}, Dept {entity['dept_id']}")
    print(f"   Predicted Weekly Sales: ${result['predictions'][0]:,.0f}")
    print(f"   Latency: {result['latency_ms']:.0f}ms")
except Exception as e:
    print(f"❌ Prediction error: {e}")

In [ ]:
# Batch predictions
print("📊 Batch Prediction Test")
print("-" * 50)

entities = [
    {"store_id": s, "dept_id": d}
    for s in [1, 10, 25, 45]
    for d in [1, 5, 10, 14]
]
print(f"Scoring {len(entities)} entities...")

try:
    result = predict_with_feast(entities)
    results_df = pd.DataFrame([
        {**e, "prediction": f"${p:,.0f}"}
        for e, p in zip(entities, result["predictions"])
    ])
    print(f"\n✅ {len(results_df)} predictions in {result['latency_ms']:.0f}ms")
    print(f"   Avg latency per prediction: {result['latency_ms']/len(entities):.1f}ms")
    print()
    display(results_df)
except Exception as e:
    print(f"❌ Batch prediction error: {e}")

---
## Pipeline Complete

### End-to-End Summary

| Notebook | Component | Output |
|----------|-----------|--------|
| 01a/01b - Features | Feast Operator | Features registered, online store materialized |
| 02 - Training | Kubeflow + DDP | Model trained with `get_historical_features()` |
| 03 - Inference | KServe | Model serving with Feast SDK integration |

### Key Benefits

| Benefit | How |
|---------|----- |
| **No feature skew** | Same FeatureService (`inference_features`) for train & serve |
| **Scalable training** | Kubeflow TrainJob distributes across nodes |
| **Scalable serving** | KServe auto-scales 1-3 replicas |
| **Low latency** | Feast online store via gRPC (<50ms feature lookup) |
| **Simple client** | Send entity IDs only, get predictions |

### Troubleshooting

| Issue | Solution |
|-------|----------|
| InferenceService not ready | Check pod logs: `oc logs -l app=sales-forecasting -n feast-trainer-demo` |
| Feast lookup fails | Verify online store: `oc get featurestore -n feast-trainer-demo` |
| Model load error | Check model artifacts exist in `/shared/models/` |
| TLS/cert errors | Verify `feast-salesforecasting-client-ca` ConfigMap mounted |

### Production Deployment

```bash
# Deploy all infrastructure
oc apply -k manifests/

# Monitor training
oc logs -f -l trainer.kubeflow.org/trainjob-name=sales-forecast

# Check serving
oc get inferenceservice -n feast-trainer-demo
```